# IFN647 Week 5 Review Question 2 (Likelihood Model) - Solution
<author>&copy; Prepared by Professor Yuefeng Li (QUT) </author>

In [10]:
#Function index_docs() returns an index - a dictionary {term:{docID1:freq1, DocID2:freq2, ...}, …}

import glob, os
import string
from stemming.porter2 import stem

def index_docs(inputpath,stop_words):
    Index = {}    # initialize the index
    os.chdir(inputpath)
    for file_ in glob.glob("*.xml"):
        start_end = False
        for line in open(file_):
            line = line.strip()
            if(start_end == False):
                if line.startswith("<newsitem "):
                    for part in line.split():
                        if part.startswith("itemid="):
                            docid = part.split("=")[1].split("\"")[1]
                            break  
                if line.startswith("<text>"):
                    start_end = True  
            elif line.startswith("</text>"):
                break
            else:
                line = line.replace("<p>", "").replace("</p>", "")
                line = line.translate(str.maketrans('','', string.digits)).translate(str.maketrans(string.punctuation, ' '*len(string.punctuation)))
                for term in line.split():
                    term = stem(term.lower())
                    if len(term) > 2 and term not in stop_words:
                        try:
                            try:
                                Index[term][docid] += 1
                            except KeyError:
                                Index[term][docid]=1
                        except KeyError:  
                            Index[term] = {docid:1} 
    return Index

In [12]:
# define a function likelihood_IR(I,Q) to estimate P(Q|D) for all documents D in a folder indexed by I

def likelihood_IR(I, Q): #index I is a dirctionary of term:(itemId:freq)
    L={}    # L is the selected inverted list
    R={}    # R is a directionary of docId:score
    D_len={} # D_len is a directionary of docId:length
    for list in I.items():
        for id in list[1].items(): 
            R[id[0]]=1       # get all document IDs and initialize as 1
            D_len[id[0]]=0.5 # a small non-zero value as a denominator
        if (list[0] in Q):     # select inverted lists based on the query Q
                L[list[0]]= I[list[0]]
    for q_term in Q.items(): # L may not contain all query terms
        if not(q_term[0] in L):
            L[q_term[0]]={}
    for list in I.items():
        for id in list[1].items(): #Count term occurrences in documents
            D_len[id[0]]= D_len[id[0]] + id[1]  
    for (d, sd) in R.items():
        for (term, f) in L.items():
            if not(d in f):
                f[d]=0
            sd = sd*(f[d]/D_len[d]) #see the equation in the lecture notes 
        R[d] = sd
    return R

In [14]:
# Test the defined functions
curr_path=os.getcwd()
stopwords_f = open('common-english-words.txt', 'r')
stop_words = stopwords_f.read().split(',')
stopwords_f.close()
Test_path = curr_path+'/Test_docs'
Index_Test = index_docs(Test_path, stop_words) #Index all terms 
os.chdir(curr_path)
Query = {'share':1, 'bank':1}
#test the likelyhood IR model
IR_result = likelihood_IR(Index_Test, Query)
y = sorted(IR_result.items(), key=lambda x: x[1],reverse=True)
print('Likelihood Query Result ---------')
for (id, w) in y:
    print('Document ID: '+id + ' and relevance weight: ' + str(w))
            

Likelihood Query Result ---------
Document ID: 807606 and relevance weight: 0.0009652315550282532
Document ID: 807600 and relevance weight: 0.00014154245894636803
Document ID: 809481 and relevance weight: 0.0
Document ID: 809495 and relevance weight: 0.0
